In [372]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import re

In [373]:
data = pd.read_csv('data/online_retail_II.csv')
data.shape
data.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Business question: - Refine Later ###
* “How does price impact demand, and what price maximizes revenue?”
*  What is the optimal price point?
*  How sensitive are customers to price changes?
*  Do promotions actually increase revenue or just volume?

### Project Outline - Refine Later ###
1. EDA
    * Price distributions
    * Demand distibution (p vs. Q)
    * Promo comperitors
2. DGP
    * Endogeniety? 
    * Simultaneity? 
    * Promo treatment effects? 
3. Cleaning/Preprocessing
4. Feature Eng
    * Demand Signals
        * Rolling Ave
    * Price in context
    * Market in context
5. Elasticity Estimation
    * log-OLS
6. Addressing Endogeneity in Pricing
    * FE estimation
    * IV estimation
7. Validation
8. RevOpt
    * Simulate Rev/Profit curves
    * Identify Max and regionality within P/Q
9. A/B testing promos
    * OlS
    * DiD
10. Elasticity by segment
11. Demand & Revenue Curve Visualization
12. Streamlit? 


### Data Description ###
* InvoiceNo: Invoice number. Nominal. A 6-digit integral number uniquely assigned to each transaction. If this code starts with the letter 'c', it indicates a cancellation.
* StockCode: Product (item) code. Nominal. A 5-digit integral number uniquely assigned to each distinct product.
* Description: Product (item) name. Nominal.
* Quantity: The quantities of each product (item) per transaction. Numeric.
* InvoiceDate: Invoice date and time. Numeric. The day and time when a transaction was generated.
* Price: Unit price. Numeric. Product price per unit in sterling (Â£).
* CustomerID: Customer number. Nominal. A 5-digit integral number uniquely assigned to each customer.
* Country: Country name. Nominal. The name of the country where a customer resides.

In [374]:
data['Refund'] = np.where(data['Price'] <= 0, 1, 0)
data['Return'] = np.where(data['Quantity'] <= 0, 1, 0 )
returned_sales = data[(data['Return'] == 1) | (data['Refund'] == 1)]


In [375]:
returned_sales['Description'].value_counts(ascending=False).head(10)

Description
Manual                                544
REGENCY CAKESTAND 3 TIER              351
POSTAGE                               229
BAKING SET 9 PIECE RETROSPOT          211
STRAWBERRY CERAMIC TRINKET BOX        186
Discount                              172
check                                 162
WHITE HANGING HEART T-LIGHT HOLDER    140
WHITE CHERRY LIGHTS                   121
RED RETROSPOT CAKE STAND              112
Name: count, dtype: int64

In [376]:
sales_data = data[(data['Refund'] == 0) & (data['Return'] == 0)]
sales_data.shape

(1041671, 10)

In [391]:
sales_data['Clean_Desc'] = (
    sales_data['Description']
    .fillna("")
    .str.upper()
    .str.replace(r"[^A-Z\s]", " ", regex = True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

product_desc = (
    sales_data
    .groupby('StockCode')['Clean_Desc']
    .agg(lambda x: ' '.join(x.unique()))
    .reset_index()
)

product_desc.head()

,StockCode,Clean_Desc
0,10002,INFLATABLE POLITICAL GLOBE
1,10002R,ROBOT PENCIL SHARPNER
2,10080,GROOVY CACTUS INFLATABLE
3,10109,BENDY COLOUR PENCILS
4,10120,DOGGY RUBBER


In [378]:
sales_data = sales_data[sales_data['Clean_Desc'] != '']
sales_data = sales_data[~sales_data['Clean_Desc'].str.contains('POSTAGE|CARRIAGE|DISCOUNT', na = False)]
sales_data = sales_data[~sales_data['StockCode'].str.contains('gift', na = False)]
sales_data = sales_data[~sales_data['StockCode'].str.contains('TEST', na = False)]
sales_data = sales_data[~sales_data['StockCode'].str.contains('ADJUST', na = False)]
sales_data = sales_data[~sales_data['StockCode'].str.contains('FEE', na = False)]
sales_data = sales_data[~sales_data['StockCode'].str.contains('CHARGE', na = False)]

sales_data = sales_data[(sales_data['StockCode'] != 'm')&(sales_data['StockCode'] != 'M')]
sales_data = sales_data[(sales_data['StockCode'] != '84016')]

In [379]:
sales_data['min_price'] = sales_data.groupby('StockCode')['Price'].transform(lambda x: x.quantile(0.01))
sales_data['max_price'] = sales_data.groupby('StockCode')['Price'].transform(lambda x: x.quantile(0.99))
sales_data['outlier_price_high'] = np.where(sales_data['Price'] > sales_data['max_price'], 1, 0)
sales_data['outlier_price_low'] = np.where(sales_data['Price'] < sales_data['min_price'], 1, 0)
outliers = sales_data[(sales_data['outlier_price_high'] == 1) | (sales_data['outlier_price_low'] == 1)]
outliers.sort_values('Price', ascending = False)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,Refund,Return,Clean_Desc,min_price,max_price,outlier_price_high,outlier_price_low
748143,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,2011-06-10 15:33:00,649.50,15098.0,United Kingdom,0,0,PICNIC BASKET WICKER PIECES,3.7500,10.7900,1,0
748132,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098.0,United Kingdom,0,0,PICNIC BASKET WICKER PIECES,3.7500,10.7900,1,0
346540,523123,84965B,BLUE KASHMIRI COFFEE TABLE,1,2010-09-20 13:23:00,129.95,14794.0,United Kingdom,0,0,BLUE KASHMIRI COFFEE TABLE,30.0000,127.8515,1,0
31273,491970,21300,ANTIQUE EDWARDIAN DRESSER,1,2009-12-14 18:03:00,104.30,NaN,United Kingdom,0,0,ANTIQUE EDWARDIAN DRESSER,12.7500,98.3215,1,0
598114,542255,22833,HALL CABINET WITH 3 DRAWERS,1,2011-01-26 16:43:00,100.00,NaN,United Kingdom,0,0,HALL CABINET WITH DRAWERS,17.4766,92.9930,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
358764,524174,21088,SET/6 FRUIT SALAD PAPER CUPS,7128,2010-09-27 16:30:00,0.08,13687.0,United Kingdom,0,0,SET FRUIT SALAD PAPER CUPS,0.1196,1.6600,0,1
695853,551291,16008,SMALL FOLDING SCISSOR(POINTED EDGE),240,2011-04-27 14:26:00,0.08,14298.0,United Kingdom,0,0,SMALL FOLDING SCISSOR POINTED EDGE,0.1140,0.2500,0,1
298942,518505,21088,SET/6 FRUIT SALAD PAPER CUPS,7128,2010-08-09 13:10:00,0.08,14277.0,France,0,0,SET FRUIT SALAD PAPER CUPS,0.1196,1.6600,0,1
65079,495194,16044,POP-ART FLUORESCENT PENS,6144,2010-01-21 15:11:00,0.06,13902.0,Denmark,0,0,POP ART FLUORESCENT PENS,0.0672,0.4200,0,1


In [380]:
sales_data = sales_data[sales_data['outlier_price_high'] == 0]


In [381]:
top_desc = sales_data['Clean_Desc'].value_counts(ascending= False).head(25)
print("Top full descriptions:")
print(top_desc)


Top full descriptions:
Clean_Desc
WHITE HANGING HEART T LIGHT HOLDER    5752
REGENCY CAKESTAND TIER                4054
JUMBO BAG RED RETROSPOT               3388
ASSORTED COLOUR BIRD ORNAMENT         2922
PARTY BUNTING                         2738
LUNCH BAG BLACK SKULL                 2482
LUNCH BAG SUKI DESIGN                 2470
STRAWBERRY CERAMIC TRINKET BOX        2409
JUMBO STORAGE BAG SUKI                2398
HEART OF WICKER SMALL                 2293
JUMBO SHOPPER VINTAGE RED PAISLEY     2263
TEATIME FAIRY CAKE CASES              2257
FRENCH BLUE METAL DOOR SIGN           2222
PAPER CHAIN KIT S CHRISTMAS           2194
REX CASH CARRY JUMBO SHOPPER          2189
LUNCH BAG SPACEBOY DESIGN             2183
HOME BUILDING BLOCK WORD              2151
WOODEN FRAME ANTIQUE WHITE            2143
LUNCH BAG CARS BLUE                   2119
NATURAL SLATE HEART CHALKBOARD        2115
WOODEN PICTURE FRAME WHITE FINISH     2085
HEART OF WICKER LARGE                 2082
PACK OF PINK PAISLEY

In [382]:
def get_words (df, col):
    all_words = (
        df[col]
        .str.split()
        .explode()
    )

    stopwords = {
     "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
     "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
    }

    word_counts = (
        all_words[
            all_words.notna()
            & (all_words.str.len() > 2)
            & (~all_words.isin(stopwords))
        ]
        .value_counts()
        .head(50)
    )
    print("\nTop single words:")
    print(word_counts)
get_words(sales_data, 'Clean_Desc')


Top single words:
Clean_Desc
BAG           91150
RED           90695
HEART         77884
PINK          63826
RETROSPOT     57287
VINTAGE       54653
DESIGN        53138
WHITE         49709
BOX           49598
METAL         44828
CAKE          44786
CHRISTMAS     43938
BLUE          41290
HANGING       36052
LIGHT         35618
SIGN          35048
JUMBO         34646
HOLDER        34112
PAPER         30528
LUNCH         29904
GLASS         26496
TEA           26262
CARD          25461
DECORATION    24718
WOODEN        23399
CASES         23149
BOTTLE        22882
SPOTTY        22111
HOT           21774
WATER         21107
ROSE          20447
CERAMIC       20186
SPACEBOY      18211
MUG           18007
PAISLEY       17874
FAIRY         17186
BLACK         16855
TIN           16820
POLKADOT      16660
CREAM         16540
HOME          16405
GREEN         16101
SPOT          16097
PARTY         15301
GARDEN        15098
FELTCRAFT     15054
MINI          14990
LOVE          14454
BUNTING   

In [383]:
stopwords = {
     "THE", "AND", "OF", "IN", "TO", "FOR", "ON", "WITH", "A", "AN",
     "SET", "PACK", "SMALL", "LARGE", "ASSORTED"
    }

def get_bigrams(text: str) -> list[str]:
    words = [w for w in text.split() if len(w) > 2 and w not in stopwords]
    return [" ".join(pair) for pair in zip(words, words[1:])]

bigram_counter = Counter()

for desc in sales_data["Clean_Desc"]:
    bigram_counter.update(get_bigrams(desc))

top_bigrams = pd.Series(dict(bigram_counter)).sort_values(ascending=False).head(50)

print("\nTop bigrams:")
print(top_bigrams)


Top bigrams:
RED RETROSPOT        29564
METAL SIGN           27862
JUMBO BAG            26410
LIGHT HOLDER         23384
LUNCH BAG            21751
HOT WATER            19636
WATER BOTTLE         19636
CAKE CASES           18935
RED SPOTTY           14750
HANGING HEART        12981
FAIRY CAKE           11354
CHARLOTTE BAG        10537
DOLLY GIRL            9831
UNION JACK            9118
VINTAGE CHRISTMAS     8649
HEART LIGHT           8576
LUNCH BOX             8153
CAKE STAND            8024
BAG PINK              7951
DRAWER KNOB           7578
BAG RED               7485
BAG VINTAGE           7438
PINK POLKADOT         7420
PLASTERS TIN          7360
BIRTHDAY CARD         7209
RETRO SPOT            7175
HAND WARMER           7089
TRINKET BOX           6890
HEART DECORATION      6785
BAG SUKI              6658
SWEET HOME            6452
HOME SWEET            6441
CHAIN KIT             6422
PAPER CHAIN           6422
RIBBON REEL           6114
ALARM CLOCK           6078
ENGLISH ROSE  

In [441]:
categories = {
    "Textiles": r"\b(SILK FAN|DOILIES|WOOLLY HOTTIE|TOWEL|CUSHION COVER|PILLOW|RIBBON|RIBBONS|THROW|FABRIC|LINEN|TEA TOWELS|RECYCLING BAG)\b",
    "Kitchen": r"\b(MUG|MUGS|COOKIE CUTTER|JELLY MOULDS|FOOD COVER|BAKING CASES|TEA GLASS|CUTLERY|TEA SET|POPCORN HOLDER|BOWLS|BAKING SET|DISH|BOWL|SPOON|SNACK TRAY|SPOONS|FORK|KNIFE|LUNCH BAG|CUP|MUG|CAKE STAND|WATER BOTTLE|TEACUP|CAKESTAND|CAKE CASES|PLATE|MILK JUG|BUTTER DISH|OVEN GLOVE|TEAPOT|KITCHEN SCALES|COOKIE CUTTERS|FRYING PAN|JAM MAKING)\b",
    "Storage": r"\b(STORAGE CUBES|MINI CASES|MINI CHEST|TIN|TINS|CONTAINER|BIN|BOX|STORAGE BAG|BASKET|BOTTLE|JAR|CRATE|HAMPER|SNACK BOXES)\b",
    "Furniture": r"\b(MIRROR|COAT RACK|CABINET|TABLE|CHAIR|SOFA|BENCH|CUSHION|STOOL|DESK|DRAWER|SHELF)\b",
    "Toys": r"\b(NESTING BOXES|PLAYING CARDS|JIGSAW|SNAKES LADDERS|GLIDERS|BINGO|MONEY BANK|PIGGY BANK|SKIPPING ROPE|SPINNING TOPS|ROCKING HORSE|WOODEN BLOCK|BLOCK LETTERS|CHILD S APRON|CHILDRENS APRON|DOLL|DOLLY|PUZZLE|GAME|BUILDING BLOCK|MARBLES|PLAYHOUSE|TOY|SNAP CARDS)\b",
    "Accessories": r"\b(GLOVES|KEY FOB|COSMETIC BAG|OVERNIGHT BAG|PURSE|SHOPPING BAG|JUMBO BAG|SHOPPER|PICNIC BAG|HAND WARMER|CHARLOTTE BAG|SHOULDER BAG|PEG BAG|TOTE BAG|AIRLINE BAG)\b",
    "Outdoors" : r"\b(FLOWER JUG|PLANT HOLDER|HAMMOCK|GARDENERS|GARDEN|PLANT LADDER|TRELLIS|PARASOL|UMBRELLA|WATERING CAN)\b",
    "Paper_goods": r"\b(SKETCHBOOK|NOTEBOOKS|PASSPORT COVER|BOOK|GIFT TAGS|LUGGAGE TAG|PAPER|STATIONERY|NAPKIN|NAPKINS|TISSUE|TISSUES|GIFT TAG|COAT HANGER|CARD|GIFT BAG|GIFT WRAP|PARTY INVITES|NOTEBOOK|JOURNAL|ENVELOPE|WRAPPING|WRAP|STICKER)\b",
    "Decor": r"\b(METAL LANTERN|CHOCOLATECANDLE|RECORD FRAME|COVER FRAME|LAMP|SCENTED CANDLES|DINNER CANDLES|DOOR HANGER|FILIGREE DOVE|HANGING METAL|CLOTHES PEGS|MEMOBOARD|MEMO BOARD|DOORSTOP|SIGN|WOODEN FRAME|CLOCK|LIGHT|LIGHTS|DOOR MAT|DOORMAT|METAL SIGN|PHOTO FRAME|PICTURE FRAME|DECORATION|BUNTING|DRAWER KNOB|HEART OF WICKER|ORNAMENT|HOOK HANGER|DOOR SIGN|CANDLEHOLDER|TRINKET POT|WALL ART|FIGURINE|STATUE)\b",
    "Crafts": r"\b(CRAYONS|BLACK BOARD|STAMP|STENCIL|COLOURING SET|FELTCRAFT|PAINT|PENCIL|PENCILS|CHALK|CHALKBOARD|PEN|MARKER|SEWING|DRAWING SLATE)\b",
    "Holidays": r"\b(CHRISTMAS|VALENTINES|THANKSGIVING|BIRTHDAY|PARTY CANDLES)\b"
}
product_desc["Category"] = "Other"

for category, pattern in categories.items():
    mask = product_desc["Clean_Desc"].str.contains(pattern, na=False, regex=True)
    product_desc.loc[mask, "Category"] = category

print("\nCategory counts:")
print(product_desc["Category"].value_counts())


Category counts:
Category
Other          1991
Decor           665
Kitchen         413
Paper_goods     407
Storage         361
Furniture       236
Holidays        220
Crafts          148
Toys            132
Outdoors        123
Accessories     121
Textiles         75
Name: count, dtype: int64


C:\Users\Kolby\AppData\Local\Temp\ipykernel_5780\2305877694.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = product_desc["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby\AppData\Local\Temp\ipykernel_5780\2305877694.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = product_desc["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby\AppData\Local\Temp\ipykernel_5780\2305877694.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask = product_desc["Clean_Desc"].str.contains(pattern, na=False, regex=True)
C:\Users\Kolby\AppData\Local\Temp\ipykernel_5780\2305877694.py:17: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To ac

In [442]:
sales_data = sales_data.drop(columns=['Category'], errors='ignore')

In [443]:

sales_data = sales_data.merge(product_desc[['StockCode', 'Category']],
    on='StockCode',
    how='left'
)
sales_data[sales_data['StockCode'] == '21216'].value_counts('Category')

Category
Storage    833
Name: count, dtype: int64

In [444]:
transaction_data = sales_data[['StockCode','InvoiceDate','Price','Quantity','Category']]
transaction_data['Week'] = pd.to_datetime(transaction_data['InvoiceDate']).dt.isocalendar().week
transaction_data['Year'] = pd.to_datetime(transaction_data['InvoiceDate']).dt.isocalendar().year

weekly_transactions = transaction_data.groupby(['Week', 'Year', 'StockCode', 'Category']).agg({'Price': 'mean', 'Quantity': 'sum'}).reset_index()

C:\Users\Kolby\AppData\Local\Temp\ipykernel_5780\467242821.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_data['Week'] = pd.to_datetime(transaction_data['InvoiceDate']).dt.isocalendar().week
C:\Users\Kolby\AppData\Local\Temp\ipykernel_5780\467242821.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_data['Year'] = pd.to_datetime(transaction_data['InvoiceDate']).dt.isocalendar().year


In [445]:
weekly_transactions = (
    transaction_data
    .groupby(['StockCode', 'Year', 'Week'], as_index=False)
    .agg(
        Price=('Price', 'mean'),
        Quantity=('Quantity', 'sum')
    )
)

product_categories = (
    transaction_data
    .groupby('StockCode')['Category']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)

all_products = weekly_transactions['StockCode'].unique()

all_periods = (
    weekly_transactions[['Year', 'Week']]
    .drop_duplicates()
    .sort_values(['Year', 'Week'])
)

panel = (
    pd.MultiIndex
    .from_product(
        [all_products, all_periods.index],
        names=['StockCode', 'period_id']
    )
    .to_frame(index=False)
)

panel = panel.merge(
    all_periods.reset_index().rename(columns={'index': 'period_id'}),
    on='period_id',
    how='left'
).drop(columns='period_id')

weekly_trans_full = panel.merge(
    weekly_transactions,
    on=['StockCode', 'Year', 'Week'],
    how='left'
)

weekly_trans_full['Quantity'] = weekly_trans_full['Quantity'].fillna(0)

weekly_trans_full = weekly_trans_full.merge(
    product_categories,
    on='StockCode',
    how='left'
)

weekly_trans_full = weekly_trans_full.sort_values(['StockCode', 'Year', 'Week'])

weekly_trans_full['Price'] = (
    weekly_trans_full
    .groupby('StockCode')['Price']
    .ffill()
    .bfill()
)

In [446]:
weekly_trans_full['Revenue'] = weekly_trans_full['Price'] * weekly_trans_full['Quantity']

weekly_trans_full.groupby('Category')['Revenue'].sum().sort_values(ascending=False)

Category
Decor          5.523519e+06
Kitchen        3.624286e+06
Other          3.237253e+06
Storage        2.821753e+06
Accessories    1.717191e+06
Paper_goods    1.577573e+06
Holidays       1.108184e+06
Crafts         9.827510e+05
Toys           8.257005e+05
Outdoors       7.657160e+05
Furniture      6.006533e+05
Textiles       4.597592e+05
Name: Revenue, dtype: float64

In [455]:
weekly_trans_full.groupby('StockCode')['Price'].var().sort_values()

StockCode
SP1002       0.000000
84593        0.000000
84582        0.000000
21406        0.000000
21409        0.000000
             ...     
84967A     525.737569
84967B     592.995071
22826     1605.666542
22655     7397.153698
22656     8875.968540
Name: Price, Length: 4892, dtype: float64